In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sentencepiece as spm
import pandas as pd
import os


# STEP 1: Load your preprocessed dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Project Phase-II/hindi_cleaned_final.csv")
print(f"Dataset shape: {df.shape}")
print(df['text'].head(3))

Dataset shape: (165041, 1)
0    एक विराम चिह्न है जिसे विस्मयादिबोधक चिह्न कहत...
1    पिक्चर्स या एंड पिक्चर्स एक टेलीविजन चैनल है ज...
2    जे॰बी॰ से ने अपनी पुस्तक ट्रेट डी एकनोमिक पोल्...
Name: text, dtype: object


# STEP 2: Save text to a plain .txt file
# (SentencePiece needs raw text file as input)

In [ ]:

with open("hindi_corpus.txt", "w", encoding="utf-8") as f:
    for line in df['text'].dropna():
        f.write(line.strip() + "\n")

print("Corpus saved to hindi_corpus.txt")

Corpus saved to hindi_corpus.txt


# STEP 3: Train SentencePiece BPE Tokenizer

In [ ]:
spm.SentencePieceTrainer.train(
    input="hindi_corpus.txt",
    model_prefix="hindi_tokenizer",   # Output: hindi_tokenizer.model + .vocab
    vocab_size=8000,                  # You can increase to 16000 or 32000
    character_coverage=0.9995,        # High coverage for Hindi Unicode chars
    model_type="bpe",                 # BPE (Byte Pair Encoding)
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>",
    user_defined_symbols=["<mask>"]
)

print("Tokenizer trained! Files: hindi_tokenizer.model, hindi_tokenizer.vocab")


Tokenizer trained! Files: hindi_tokenizer.model, hindi_tokenizer.vocab


# STEP 4: Load and Test the Tokenizer

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("hindi_tokenizer.model")

sample = "भारत एक महान देश है।"
tokens = sp.encode(sample, out_type=str)
ids    = sp.encode(sample, out_type=int)

print(f"\nSample text : {sample}")
print(f"Tokens      : {tokens}")
print(f"Token IDs   : {ids}")
print(f"Decoded     : {sp.decode(ids)}")
print(f"Vocab size  : {sp.get_piece_size()}")


Sample text : भारत एक महान देश है।
Tokens      : ['▁भारत', '▁एक', '▁महान', '▁देश', '▁है', '।']
Token IDs   : [72, 50, 1637, 592, 15, 7960]
Decoded     : भारत एक महान देश है।
Vocab size  : 8000


# Convert entire dataset to token ID sequences

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 128   # We can use 256 or 512 if GPU allows

def tokenize_dataset(df, sp_model, max_len=MAX_SEQ_LEN):
    all_ids = []
    for text in df['text'].dropna():
        ids = sp_model.encode(text, out_type=int)
        # Add BOS and EOS tokens
        ids = [sp_model.bos_id()] + ids + [sp_model.eos_id()]
        all_ids.extend(ids)
    return all_ids

all_token_ids = tokenize_dataset(df, sp)
print(f"Total tokens in dataset: {len(all_token_ids)}")

# Create sliding window sequences for LM training
class HindiTextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.seq_len = seq_len
        # Chunk into non-overlapping blocks of seq_len + 1
        self.examples = []
        for i in range(0, len(token_ids) - seq_len, seq_len):
            chunk = token_ids[i : i + seq_len + 1]
            if len(chunk) == seq_len + 1:
                self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        chunk = self.examples[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:],  dtype=torch.long)
        return x, y

dataset    = HindiTextDataset(all_token_ids, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Total sequences (batches): {len(dataset)}")

x_sample, y_sample = dataset[0]
print(f"Input shape : {x_sample.shape}")
print(f"Target shape: {y_sample.shape}")

Total tokens in dataset: 69719428
Total sequences (batches): 544683
Input shape : torch.Size([128])
Target shape: torch.Size([128])


# Simple GPT-style Transformer Model

In [ ]:
import torch
import torch.nn as nn

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        # Causal mask: upper triangle = -inf so future tokens are blocked
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        )

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x, attn_mask=self.mask[:x.size(1), :x.size(1)])
        return attn_out


class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len, ff_dim):
        super().__init__()
        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, seq_len)
        self.ln2  = nn.LayerNorm(embed_dim)
        self.ff   = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8,
                 num_layers=4, seq_len=128, ff_dim=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb   = nn.Embedding(seq_len, embed_dim)
        self.blocks    = nn.Sequential(
            *[GPTBlock(embed_dim, num_heads, seq_len, ff_dim)
              for _ in range(num_layers)]
        )
        self.ln_f  = nn.LayerNorm(embed_dim)
        self.head  = nn.Linear(embed_dim, vocab_size, bias=False)
        self.seq_len = seq_len

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(positions)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)       # (B, T, vocab_size)
        return logits


VOCAB_SIZE = sp.get_piece_size()    # 8000
model = MiniGPT(vocab_size=VOCAB_SIZE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 6,237,696


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)   # ignore <pad>
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for step, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        logits = model(x)                          # (B, T, vocab_size)
        loss   = criterion(
            logits.view(-1, VOCAB_SIZE),           # (B*T, vocab_size)
            y.view(-1)                             # (B*T,)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_loss:.4f}\n")
    scheduler.step()

    # Save checkpoint every epoch
    torch.save({
        'epoch'            : epoch,
        'model_state_dict' : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss'             : avg_loss,
    }, f"checkpoint_epoch_{epoch+1}.pt")

print("Training complete!")

Epoch 1 | Step 0 | Loss: 9.1475
Epoch 1 | Step 100 | Loss: 7.3466
Epoch 1 | Step 200 | Loss: 6.8127
Epoch 1 | Step 300 | Loss: 6.6666
Epoch 1 | Step 400 | Loss: 6.6530
Epoch 1 | Step 500 | Loss: 6.6226
Epoch 1 | Step 600 | Loss: 6.4188
Epoch 1 | Step 700 | Loss: 6.4154
Epoch 1 | Step 800 | Loss: 6.3976
Epoch 1 | Step 900 | Loss: 6.3075
Epoch 1 | Step 1000 | Loss: 6.1800
Epoch 1 | Step 1100 | Loss: 5.9770
Epoch 1 | Step 1200 | Loss: 5.9814
Epoch 1 | Step 1300 | Loss: 5.8220
Epoch 1 | Step 1400 | Loss: 6.0171
Epoch 1 | Step 1500 | Loss: 5.9426
Epoch 1 | Step 1600 | Loss: 6.1031
Epoch 1 | Step 1700 | Loss: 5.5646
Epoch 1 | Step 1800 | Loss: 5.9602
Epoch 1 | Step 1900 | Loss: 5.5082
Epoch 1 | Step 2000 | Loss: 5.8200
Epoch 1 | Step 2100 | Loss: 5.6442
Epoch 1 | Step 2200 | Loss: 5.6289
Epoch 1 | Step 2300 | Loss: 5.4751
Epoch 1 | Step 2400 | Loss: 5.3391
Epoch 1 | Step 2500 | Loss: 5.5985
Epoch 1 | Step 2600 | Loss: 5.2248
Epoch 1 | Step 2700 | Loss: 5.5878
Epoch 1 | Step 2800 | Loss: 5.62

# Evaluation - Perplexity

In [ ]:
import torch
import math

def evaluate(model, dataloader, criterion, device, vocab_size):
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss   = criterion(
                logits.view(-1, vocab_size),
                y.view(-1)
            )

            total_loss   += loss.item() * y.numel()
            total_tokens += y.numel()

    avg_loss    = total_loss / total_tokens
    perplexity  = math.exp(avg_loss)
    return avg_loss, perplexity

criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

avg_loss, perplexity = evaluate(model, dataloader, criterion, device, VOCAB_SIZE)

print(f" Evaluation Complete!")
print(f" Average Loss : {avg_loss:.4f}")
print(f" Perplexity   : {perplexity:.2f}")


# Inference - Next Word Prediction


In [ ]:
import torch
def generate_text(model, sp, prompt, max_new_tokens=50, temperature=1.0, device='cpu'):
    model.eval()
    input_ids = sp.encode(prompt, out_type=int)
    input_ids = [sp.bos_id()] + input_ids
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    generated = input_ids.copy()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_trimmed = torch.tensor(
                [generated[-model.seq_len:]], dtype=torch.long
            ).to(device)

            logits = model(input_trimmed)
            next_token_logits = logits[0, -1, :]
            next_token_logits = next_token_logits / temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            if next_token == sp.eos_id():
                break

            generated.append(next_token)
    output_ids = generated[1:]
    output_text = sp.decode(output_ids)
    return output_text
prompts = [
    "भारत एक",
    "हिंदी भाषा",
    "दिल्ली में",
]
print("=" * 50)
print("        MODEL INFERENCE - HINDI TEXT")
print("=" * 50)

for prompt in prompts:
    result = generate_text(
        model=model,
        sp=sp,
        prompt=prompt,
        max_new_tokens=30,
        temperature=0.8,
        device=device
    )
    print(f"\nInput  : {prompt}")
    print(f"Output : {result}")
    print("-" * 50)